# 01 — Data Cleaning & Validation
Python cleans and validates the source data only.


## Cell 1 — Imports

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)


## Cell 2 — Load source data

In [3]:
FILE_PATH = "D:\\New folder\\project\\Global-Superstore-Analysis\\data\\raw\\global_superstore_2016.xlsx"

orders = pd.read_excel(FILE_PATH, sheet_name="Orders")
returns = pd.read_excel(FILE_PATH, sheet_name="Returns")
people = pd.read_excel(FILE_PATH, sheet_name="People")

print("Orders:", orders.shape)
print("Returns:", returns.shape)
print("People:", people.shape)


Orders: (51290, 24)
Returns: (1079, 3)
People: (24, 2)


## Cell 3 — Dataset overview (shape, dtypes, memory)

In [4]:
for name, df in [("ORDERS", orders), ("RETURNS", returns), ("PEOPLE", people)]:
    print(f"\n===== {name} =====")
    print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
    print(f"Memory: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")
    print(df.dtypes)



===== ORDERS =====
Shape: 51,290 rows x 24 columns
Memory: 49672.5 KB
Row ID                     int64
Order ID                     str
Order Date        datetime64[us]
Ship Date         datetime64[us]
Ship Mode                    str
Customer ID                  str
Customer Name                str
Segment                      str
Postal Code              float64
City                         str
State                        str
Country                      str
Region                       str
Market                       str
Product ID                   str
Category                     str
Sub-Category                 str
Product Name                 str
Sales                    float64
Quantity                   int64
Discount                 float64
Profit                   float64
Shipping Cost            float64
Order Priority               str
dtype: object

===== RETURNS =====
Shape: 1,079 rows x 3 columns
Memory: 195.8 KB
Returned    str
Order ID    str
Region      str
dtype: 

## Cell 4 — Missing-value audit

In [5]:
for name, df in [("ORDERS", orders), ("RETURNS", returns), ("PEOPLE", people)]:
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    print(f"\n===== {name} — columns with nulls =====")
    if nulls.empty:
        print("None")
    else:
        for col, n in nulls.items():
            print(f"{col}: {n:,} nulls ({n/len(df)*100:.1f}%)")



===== ORDERS — columns with nulls =====
Postal Code: 41,296 nulls (80.5%)

===== RETURNS — columns with nulls =====
None

===== PEOPLE — columns with nulls =====
None


**Finding:** `Postal Code` shows ~80% nulls in Orders.


In [ ]:
non_us_null_rate = orders.loc[orders["Country"] != "United States", "Postal Code"].isna().mean()
us_null_rate = orders.loc[orders["Country"] == "United States", "Postal Code"].isna().mean()
print(f"Postal Code null rate OUTSIDE the US: {non_us_null_rate*100:.1f}%")
print(f"Postal Code null rate WITHIN the US:  {us_null_rate*100:.1f}%")



Postal Code null rate OUTSIDE the US: 100.0%
Postal Code null rate WITHIN the US:  0.0%

Conclusion: Postal Code is a US-only field. This is expected, not a data quality issue.


## Cell 5 — Duplicate & key audit (transaction grain)

In [7]:
print("Full-row duplicates in Orders:", orders.duplicated().sum())
print("Full-row duplicates in Returns:", returns.duplicated().sum())
print("Full-row duplicates in People:", people.duplicated().sum())

print("\n--- Grain check ---")
print("Row ID unique?", orders["Row ID"].is_unique, f"({orders['Row ID'].nunique():,} unique / {len(orders):,} rows)")
print("Order ID unique?", orders["Order ID"].is_unique,
      f"({orders['Order ID'].nunique():,} distinct orders across {len(orders):,} line items — repeats ARE expected)")


Full-row duplicates in Orders: 0
Full-row duplicates in Returns: 0
Full-row duplicates in People: 0

--- Grain check ---
Row ID unique? True (51,290 unique / 51,290 rows)
Order ID unique? False (25,728 distinct orders across 51,290 line items — repeats ARE expected)


## Cell 6 — Referential integrity: Returns → Orders

In [8]:
returns_in_orders = returns["Order ID"].isin(orders["Order ID"])
match_rate = returns_in_orders.mean()
print(f"Returns Order IDs found in Orders: {match_rate*100:.2f}% ({returns_in_orders.sum():,} / {len(returns):,})")

assert match_rate == 1.0, "FAIL: some Return Order IDs do not exist in Orders — investigate before proceeding."
print("PASS: every Return Order ID exists in Orders.")


Returns Order IDs found in Orders: 100.00% (1,079 / 1,079)
PASS: every Return Order ID exists in Orders.


## Cell 7 — Region mismatch: Orders ↔ People (detect only, do not fix yet)

In [9]:
orders_regions = set(orders["Region"].unique())
people_regions = set(people["Region"].unique())

print("Regions in Orders:", len(orders_regions))
print("Regions in People:", len(people_regions))
print("In Orders but not People:", orders_regions - people_regions)
print("In People but not Orders:", people_regions - orders_regions)


Regions in Orders: 23
Regions in People: 24
In Orders but not People: {'Canada'}
In People but not Orders: {'Eastern Canada', 'Western Canada'}


## Cell 8 — Business-rule validation

In [10]:
checks = {
    "Sales >= 0": (orders["Sales"] >= 0).all(),
    "Quantity > 0": (orders["Quantity"] > 0).all(),
    "Discount between 0 and 1": orders["Discount"].between(0, 1).all(),
    "Shipping Cost >= 0": (orders["Shipping Cost"] >= 0).all(),
    "Ship Date >= Order Date": (orders["Ship Date"] >= orders["Order Date"]).all(),
    "Order ID not null": orders["Order ID"].notna().all(),
    "Customer ID not null": orders["Customer ID"].notna().all(),
    "Product ID not null": orders["Product ID"].notna().all(),
}

for rule, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'} — {rule}")

assert all(checks.values()), "One or more business-rule checks failed — see above."


PASS — Sales >= 0
PASS — Quantity > 0
PASS — Discount between 0 and 1
PASS — Shipping Cost >= 0
PASS — Ship Date >= Order Date
PASS — Order ID not null
PASS — Customer ID not null
PASS — Product ID not null


## Cell 9 — Customer identity


In [11]:
n_ids = orders["Customer ID"].nunique()
n_names = orders["Customer Name"].nunique()
print(f"Unique Customer IDs: {n_ids:,}")
print(f"Unique Customer Names: {n_names:,}")

name_country_spread = orders.groupby("Customer Name")["Country"].nunique()
print(f"\nAvg countries per Customer Name: {name_country_spread.mean():.1f}")
print(f"Max countries for a single name: {name_country_spread.max()}")


Unique Customer IDs: 17,415
Unique Customer Names: 796

Avg countries per Customer Name: 19.7
Max countries for a single name: 32


## Cell 10 — Region

In [12]:
region_mapping = {
    "Eastern Canada": "Canada",
    "Western Canada": "Canada",
}

people["Region"] = people["Region"].replace(region_mapping)

# Collapse to ONE row per region — combine manager names where more than one exists
people_clean = (
    people.groupby("Region")["Person"]
    .apply(lambda names: " / ".join(sorted(names)))
    .reset_index()
)
people_clean.columns = ["Region", "Manager"]

print("People rows before collapse:", len(people))
print("People rows after collapse:", len(people_clean), "(expected 23, down from 24)")

# Re-verify the join now resolves cleanly
unmatched = set(orders["Region"].unique()) - set(people_clean["Region"].unique())
print("Regions still unmatched:", unmatched if unmatched else "NONE")
assert len(unmatched) == 0, "Region mapping did not fully resolve — investigate."
print("PASS: every Order region now maps to exactly one manager row.")


People rows before collapse: 24
People rows after collapse: 23 (expected 23, down from 24)
Regions still unmatched: NONE
PASS: every Order region now maps to exactly one manager row.


## Cell 11 — Whitespace check 


In [13]:
before_count = (orders["Product Name"].str.strip() != orders["Product Name"]).sum()
print(f"Before: {before_count} rows have leading/trailing whitespace in Product Name")

orders["Product Name"] = orders["Product Name"].str.strip()

after_count = (orders["Product Name"].str.strip() != orders["Product Name"]).sum()
print(f"After:  {after_count} rows have leading/trailing whitespace in Product Name")


Before: 16 rows have leading/trailing whitespace in Product Name
After:  0 rows have leading/trailing whitespace in Product Name


## Cell 12 — Basic reusable fields
Only simple, non-business-logic derived fields belong here. `Delivery Days` is
plain date arithmetic. `Is Returned` is needed to validate the returns
relationship, not to calculate a business return-rate metric — that rate
(correctly computed at the distinct-order grain) is calculated in SQL next.

In [14]:
orders["Delivery Days"] = (orders["Ship Date"] - orders["Order Date"]).dt.days
orders["Is Returned Line"] = orders["Order ID"].isin(returns["Order ID"])

print("Delivery Days range:", orders["Delivery Days"].min(), "-", orders["Delivery Days"].max())

# Sanity-check only — NOT the final business metric. True return rate (by
# distinct order) will be calculated in SQL against the loaded fact table.
line_level_check = orders["Is Returned Line"].mean() * 100
order_level_check = orders.loc[orders["Is Returned Line"], "Order ID"].nunique() / orders["Order ID"].nunique() * 100
print(f"\nSanity check only:")
print(f"  Line-level mean (NOT the metric to report): {line_level_check:.2f}%")
print(f"  Order-level rate (the correct metric, will be recomputed in SQL): {order_level_check:.2f}%")


Delivery Days range: 0 - 7

Sanity check only:
  Line-level mean (NOT the metric to report): 4.33%
  Order-level rate (the correct metric, will be recomputed in SQL): 4.19%


## Cell 13 — Final validation after cleaning

In [15]:
print("Re-checking business rules after cleaning:")
final_checks = {
    "Sales >= 0": (orders["Sales"] >= 0).all(),
    "Quantity > 0": (orders["Quantity"] > 0).all(),
    "Discount between 0 and 1": orders["Discount"].between(0, 1).all(),
    "Ship Date >= Order Date": (orders["Ship Date"] >= orders["Order Date"]).all(),
    "No whitespace in Product Name": (orders["Product Name"].str.strip() == orders["Product Name"]).all(),
    "Region mapping resolves": len(set(orders["Region"].unique()) - set(people_clean["Region"].unique())) == 0,
}
for rule, passed in final_checks.items():
    print(f"{'PASS' if passed else 'FAIL'} — {rule}")
assert all(final_checks.values())

print("\nFinal shapes:")
print("Orders:", orders.shape)
print("Returns:", returns.shape)
print("People (cleaned):", people_clean.shape)


Re-checking business rules after cleaning:
PASS — Sales >= 0
PASS — Quantity > 0
PASS — Discount between 0 and 1
PASS — Ship Date >= Order Date
PASS — No whitespace in Product Name
PASS — Region mapping resolves

Final shapes:
Orders: (51290, 26)
Returns: (1079, 3)
People (cleaned): (23, 2)


## Cell 14 — Save cleaned outputs


In [16]:
orders.to_csv("../outputs/orders_clean.csv", index=False)
returns.to_csv("../outputs/returns_clean.csv", index=False)
people_clean.to_csv("../outputs/people_clean.csv", index=False)

print("Saved: orders_clean.csv, returns_clean.csv, people_clean.csv -> outputs/")


Saved: orders_clean.csv, returns_clean.csv, people_clean.csv -> outputs/
